# Pixel Art SDXL LoRA (A100 <20h)

End-to-end pipeline: auto-download -> ingest -> clean -> caption -> train -> eval.

In [1]:
  !git clone https://github.com/arifdag/Text-to-Image-Pixel-Art.git
  %cd /content/Text-to-Image-Pixel-Art
  !git pull origin main
  !ls

Cloning into 'Text-to-Image-Pixel-Art'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 41 (delta 3), reused 40 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 35.53 KiB | 11.84 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/Text-to-Image-Pixel-Art
From https://github.com/arifdag/Text-to-Image-Pixel-Art
 * branch            main       -> FETCH_HEAD
Already up to date.
AGENTS.md  data		    pyproject.toml  requirements-dev.txt  tests
artifacts  DATA_SOURCES.md  README.md	    requirements.txt
configs    notebooks	    Report.md	    src


In [2]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Fri Feb 20 21:34:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
!pip install -q -U pip
!pip install -q --upgrade torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -r requirements.txt -r requirements-dev.txt
!pip install -q -U datasets pyyaml "pillow<12"
!pip install -q -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pixelart-generator (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.5.1+cu121 which is incompatible.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.5.1+cu121 which is incompatible.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Optional: Clone Diffusers Examples
Required for the SDXL LoRA training script path in `configs/train_sdxl_lora.yaml`.

In [ ]:
!git clone https://github.com/huggingface/diffusers.git /content/diffusers

## Step 1: Auto-download datasets and generate `configs/data_sources.yaml`

This also creates caption sidecars so you do not manually write prompts.

In [ ]:
from datasets import load_dataset
from pathlib import Path
import json
import os
import yaml

if Path('configs').exists():
    ROOT = Path('.').resolve()
elif Path('../configs').exists():
    ROOT = Path('..').resolve()
else:
    raise RuntimeError('Run this notebook from repo root or from notebooks/.')

os.chdir(ROOT)
DATA_ROOT = Path('/content/data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
(ROOT / 'configs').mkdir(parents=True, exist_ok=True)

def pick_col(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

def export_hf_dataset(dataset_id, source_id, max_items, license_name, attribution, notes):
    ds = load_dataset(dataset_id, split='train')
    cols = ds.column_names
    image_col = pick_col(cols, ['image', 'img', 'pixel_art', 'pixelart'])
    text_col = pick_col(cols, ['prompt', 'text', 'caption', 'description'])

    if image_col is None:
        raise RuntimeError(f'{dataset_id}: no image column found. columns={cols}')

    img_dir = DATA_ROOT / source_id / 'images'
    cap_path = DATA_ROOT / source_id / 'captions.jsonl'
    img_dir.mkdir(parents=True, exist_ok=True)

    n = 0
    with cap_path.open('w', encoding='utf-8') as fout:
        for row in ds:
            if max_items and n >= max_items:
                break
            img = row[image_col]
            if img is None:
                continue
            file_name = f'{n:06d}.png'
            img.save(img_dir / file_name)
            text = ''
            if text_col and row.get(text_col) is not None:
                text = str(row[text_col]).strip()
            fout.write(json.dumps({'file_name': file_name, 'text': text}, ensure_ascii=True) + '\n')
            n += 1

    return {
        'id': source_id,
        'local_path': str(img_dir),
        'captions_path': str(cap_path),
        'captions_key_field': 'file_name',
        'captions_text_field': 'text',
        'license': license_name,
        'attribution': attribution,
        'notes': notes,
        'max_files': n,
    }, n

sources = []

s1, n1 = export_hf_dataset(
    dataset_id='nerijs/pixelparti-128-v0.1',
    source_id='pixelparti_seed',
    max_items=3000,
    license_name='CC0-1.0',
    attribution='nerijs/pixelparti-128-v0.1',
    notes='Synthetic prompt-paired pixel-art starter dataset.'
)
sources.append(s1)
print('pixelparti_seed:', n1)

try:
    s2, n2 = export_hf_dataset(
        dataset_id='Limbicnation/pixel-art-character',
        source_id='pixel_character',
        max_items=2000,
        license_name='Apache-2.0',
        attribution='Limbicnation/pixel-art-character',
        notes='Character sprite dataset.'
    )
    sources.append(s2)
    print('pixel_character:', n2)
except Exception as e:
    print('Skipping optional source 2:', e)

cfg = {'sources': sources}
cfg_path = ROOT / 'configs' / 'data_sources.yaml'
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=False), encoding='utf-8')
print('Wrote:', cfg_path)
print(cfg_path.read_text(encoding='utf-8'))

## Step 2: Ingest, clean, and caption metadata

Captions from the downloaded sidecars are used automatically.

In [ ]:
!python -m pixelart.data_ingest --config configs/data_sources.yaml --output-dir data/raw --index-out data/raw/index.jsonl
!python -m pixelart.data_clean --input-dir data/raw --index-in data/raw/index.jsonl --output-dir data/clean/images --index-out data/clean/index.jsonl
!python -m pixelart.caption --index-in data/clean/index.jsonl --output-dir data/train --metadata-out data/train/metadata.jsonl --prefer-source-prompt

## Optional: Fill missing prompts with BLIP-2 captions

In [ ]:
!python -m pixelart.caption --index-in data/clean/index.jsonl --output-dir data/train --metadata-out data/train/metadata.jsonl --prefer-source-prompt --use-blip

## Step 3: Train LoRA
The config uses `resume_from_checkpoint: latest`.

In [ ]:
!python -m pixelart.train --config configs/train_sdxl_lora.yaml

## Step 4: Evaluate baseline vs LoRA

In [ ]:
!python -m pixelart.eval --config configs/eval.yaml

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

grid_files = sorted(Path('artifacts/eval/grids').glob('*.png'))
if grid_files:
    img = Image.open(grid_files[0])
    plt.figure(figsize=(12, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('No grids found yet.')